# Formula 1 Standing Predictor — Modular Pipeline

This notebook demonstrates the refactored, modular functions for the F1 Standing Predictor.
All exploratory code has been organized into reusable modules:
- `predictor.data_pipeline`: FastF1 data collection, feature engineering & EWMA formulas
- `predictor.model`: Data cleaning, train/test splitting, XGBRanker training & evaluation
- `predictor.predict_service`: Intelligent race data lookup & prediction service
- `predictor.server`: REST API server for the web frontend

In [3]:
import sys
import os

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from predictor.model import (
    load_and_clean_data,
    split_data,
    train_ranker,
    evaluate_model,
    save_model,
    load_model
)
from predictor.predict_service import (
    get_race_data,
    predict_session_standings,
    format_standings_display,
    export_predictions_json,
    get_available_races
)

## 1. Load & Clean Dataset

In [4]:
cleaned_df = load_and_clean_data("data/driver_results_final.csv")
print("Cleaned dataset shape:", cleaned_df.shape)
cleaned_df.head()

Cleaned dataset shape: (3766, 12)


,year,session_key,driver_id,air_temp,rainfall,wind_speed,quali_relative_time,driver_points,constructor_points,constructors_ewma,driver_ewma,relevance_score
0,2018,5066,vettel,24.077477,0.045045,3.691892,674.0,25.0,40.0,0.0,0.0,22.0
1,2018,5066,hamilton,24.077477,0.045045,3.691892,0.0,18.0,22.0,0.0,0.0,21.0
2,2018,5066,raikkonen,24.077477,0.045045,3.691892,664.0,15.0,40.0,0.0,0.0,20.0
3,2018,5066,ricciardo,24.077477,0.045045,3.691892,988.0,12.0,20.0,0.0,0.0,19.0
4,2018,5066,alonso,24.077477,0.045045,3.691892,2433.0,10.0,12.0,0.0,0.0,18.0


## 2. Split into Train (< 2024) and Test (>= 2024)

In [5]:
splits = split_data(cleaned_df, split_year=2024)
X_train, y_train, qid_train = splits['X_train'], splits['y_train'], splits['qid_train']
X_test, y_test, qid_test = splits['X_test'], splits['y_test'], splits['qid_test']
test_df = splits['test_df']

print(f"Train samples: {len(X_train)} across {qid_train.nunique()} sessions")
print(f"Test samples:  {len(X_test)} across {qid_test.nunique()} sessions")

Train samples: 2500 across 125 sessions
Test samples:  1266 across 62 sessions


## 3. Train XGBRanker with Test Set Evaluation Integration

In [6]:
ranker = train_ranker(
    X_train,
    y_train,
    qid_train,
    X_test=X_test,
    y_test=y_test,
    qid_test=qid_test,
    learning_rate=0.05,
    max_depth=6,
    n_estimators=100
)
save_model(ranker, "ranker_model.json")

Model saved to ranker_model.json


## 4. Comprehensive Model Evaluation on X_test
Computes NDCG@5, NDCG@10, Spearman Rank Correlation, P1 Winner Accuracy, and Podium Recall.

In [7]:
metrics = evaluate_model(ranker, X_test, y_test, qid_test, test_df)

print("=" * 50)
print("TEST SET EVALUATION RESULTS:")
print(f"  • Total Test Sessions Evaluated: {metrics['num_sessions_evaluated']}")
print(f"  • Mean NDCG@5:                   {metrics['mean_ndcg_at_5']:.4f}")
print(f"  • Mean NDCG@10:                  {metrics['mean_ndcg_at_10']:.4f}")
print(f"  • Mean Spearman Correlation:     {metrics['mean_spearman_correlation']:.4f}")
print(f"  • P1 Winner Accuracy:            {metrics['p1_winner_matches']}/{metrics['num_sessions_evaluated']} ({metrics['p1_winner_accuracy_pct']:.1f}%)")
print(f"  • Mean Podium Recall:            {metrics['mean_podium_recall_pct']:.1f}%")
print("=" * 50)

print("\nFeature Importances:")
for feat, imp in metrics['feature_importances'].items():
    print(f"  {feat:22s}: {imp:.4f}")

TEST SET EVALUATION RESULTS:
  • Total Test Sessions Evaluated: 62
  • Mean NDCG@5:                   0.9242
  • Mean NDCG@10:                  0.9124
  • Mean Spearman Correlation:     0.7090
  • P1 Winner Accuracy:            30/62 (48.4%)
  • Mean Podium Recall:            62.4%

Feature Importances:
  air_temp              : 0.0404
  rainfall              : 0.0498
  wind_speed            : 0.0433
  quali_relative_time   : 0.3064
  driver_points         : 0.3865
  constructor_points    : 0.0594
  constructors_ewma     : 0.0685
  driver_ewma           : 0.0457


## 5. Intelligent Race Prediction (CSV Cache or Dynamic FastF1 Fetch)
Predicts standings for any Grand Prix (e.g. 2026 Italian GP or 2024 Bahrain GP).

In [8]:
# Example: Italian Grand Prix (Session 11361)
race_df, metadata = get_race_data(session_key=11361, csv_path="data/driver_results_final.csv")
ranked_df = predict_session_standings(ranker, race_df)
standings = format_standings_display(ranked_df)

import pandas as pd
display_df = pd.DataFrame(standings)[
    ['predicted_position', 'full_name', 'abbreviation', 'team_id', 'prediction_score', 'actual_position', 'accuracy_delta']
]
display_df.head(22)

,predicted_position,full_name,abbreviation,team_id,prediction_score,actual_position,accuracy_delta
0,1,Kimi Antonelli,ANT,mercedes,2.1097,1,0
1,2,Lewis Hamilton,HAM,ferrari,1.0655,6,4
2,3,Charles Leclerc,LEC,ferrari,0.9824,22,19
3,4,Pierre Gasly,GAS,alpine,0.8159,7,3
4,5,George Russell,RUS,mercedes,0.7713,2,-3
5,6,Lando Norris,NOR,mclaren,0.7243,4,-2
6,7,Max Verstappen,VER,red_bull,0.7168,3,-4
7,8,Oscar Piastri,PIA,mclaren,0.6592,5,-3
8,9,Liam Lawson,LAW,red_bull,0.2696,14,5
9,10,Arvid Lindblad,LIN,rb,0.2482,8,-2


## 6. Export Predictions to JSON for the Web Application

In [9]:
export_data = export_predictions_json(
    ranker=ranker,
    csv_path="data/driver_results_final.csv",
    output_path="predictions.json",
    web_public_path="../predictions.json"
)
print(f"Exported {len(export_data['races'])} race predictions for web frontend.")

Exported 62 race predictions to predictions.json
Synced predictions to ../predictions.json
Exported 62 race predictions for web frontend.
